# Fine-Tune a Pre-trained DistilBERT Model on a Custom Dataset

In [1]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    pipeline
)

from datasets import Dataset
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score

data = {
    "text": [
        "This movie was amazing",
        "I loved the acting",
        "Fantastic story",
        "Great experience",
        "Excellent film",
        "Wonderful movie",
        "Absolutely loved it",
        "Best movie ever",
        "Highly recommended",
        "Superb performance",
        "Terrible movie",
        "Worst film ever",
        "Very boring",
        "Waste of time",
        "Bad acting",
        "Awful experience",
        "Poor storyline",
        "Not worth watching",
        "Disappointing movie",
        "Horrible film"
    ],
    "label": [
        1,1,1,1,1,1,1,1,1,1,
        0,0,0,0,0,0,0,0,0,0
    ]
}

df = pd.DataFrame(data)

dataset = Dataset.from_pandas(df)

model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

dataset = dataset.map(tokenize_function, batched=True)

dataset = dataset.train_test_split(test_size=0.2)

train_dataset = dataset["train"]
test_dataset = dataset["test"]

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=1
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

print("Off-the-Shelf Model Predictions")

base_classifier = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

sample_text = "This movie was absolutely amazing"

print(base_classifier(sample_text))

trainer.train()

predictions = trainer.predict(test_dataset)

y_pred = np.argmax(predictions.predictions, axis=1)
y_true = test_dataset["label"]

accuracy = accuracy_score(y_true, y_pred)

print("\nFine-Tuned Model Accuracy:", accuracy)

trainer.save_model("./fine_tuned_distilbert")

fine_tuned_classifier = pipeline(
    "sentiment-analysis",
    model="./fine_tuned_distilbert",
    tokenizer=tokenizer
)

print("\nFine-Tuned Model Prediction")

print(fine_tuned_classifier(sample_text))

print("\nComparison")

print("Off-the-Shelf:", base_classifier(sample_text))
print("Fine-Tuned   :", fine_tuned_classifier(sample_text))

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Off-the-Shelf Model Predictions


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

...

C:\Users\bandi\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



Fine-Tuned Model Accuracy: 1.0


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


Fine-Tuned Model Prediction
[{'label': 'LABEL_1', 'score': 0.8043739199638367}]

Comparison
Off-the-Shelf: [{'label': 'POSITIVE', 'score': 0.9998805522918701}]
Fine-Tuned   : [{'label': 'LABEL_1', 'score': 0.8043739199638367}]
